# 02 - LLaMEA Algorithm Evolution & Analysis

This notebook focuses on **inspecting prompts**, **configuring LLM providers**, **running LLaMEA evolutionary search** (in clean or noisy mode), **visualizing convergence curves**, and **cross-evaluating generated algorithms**.

In [1]:
# Setup paths and imports
import sys
import json
from pathlib import Path
import plotly.graph_objects as go

src_dir = Path.cwd()
while src_dir.name != "AAD_LLM" and src_dir.parent != src_dir:
    src_dir = src_dir.parent
src_path = str(src_dir / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)

from aad_llm.problems.bbob import BBOBProblem
from aad_llm.llm.prompts import TASK_PROMPT_NOISY, TASK_PROMPT_CLEAN, EXAMPLE_PROMPT, FORMAT_PROMPT
from aad_llm.llm.providers import build_llm, Provider
from aad_llm.core.runner import run_evolution_for_problem, run_cross_evaluation
from aad_llm.analysis.results import save_summary


## 1. Inspect System Prompts
Review the prompt templates passed to the LLM during LLaMEA evolution. Note that bounds are fetched dynamically from the target problem.

In [2]:
sample_problem = BBOBProblem(problem_id=1, dim=3)

print("=== NOISY TASK PROMPT (BBOB f1, 3D) ===")
print(TASK_PROMPT_NOISY.format(
    problem_id=sample_problem.problem_id,
    dim=sample_problem.dim,
    lower_bound=sample_problem.lower_bound,
    upper_bound=sample_problem.upper_bound
))
print("\n=== CLEAN TASK PROMPT (BBOB f1, 3D) ===")
print(TASK_PROMPT_CLEAN.format(
    problem_id=sample_problem.problem_id,
    dim=sample_problem.dim,
    lower_bound=sample_problem.lower_bound,
    upper_bound=sample_problem.upper_bound
))
print("\n=== FORMAT PROMPT ===")
print(FORMAT_PROMPT)


=== NOISY TASK PROMPT (BBOB f1, 3D) ===

You are a highly skilled computer scientist and an expert in meta-heuristic optimization.
Your task is to design a novel, continuous black-box optimization algorithm that is highly specialized for a specific target landscape (BBOB Problem ID: 1).
Critically, the objective function you are optimizing contains statistical noise. Your algorithm must be resilient to this noise and avoid being trapped by false gradients.

This is NOT a general-purpose solver. You are designing a bespoke algorithm tailored to exploit the specific features of this single noisy landscape.

Write the Python code for a class that contains a `__call__(self, problem, budget)` method.
The `problem` is the noisy objective function to be minimized. You can evaluate a point `x` by calling `y = problem(x)`, where `x` is a numpy array and `y` is a noisy float.
The `budget` is the maximum number of times you can evaluate `problem`.
The domain bounds for the search space are [-5.0,

## 2. LLM Provider Configuration
Configure LLM provider (e.g. Gemini or LM Studio) from environment variables or `.env` file.

In [3]:
import os
from dotenv import load_dotenv

env_path = src_dir / ".env"
if env_path.exists():
    load_dotenv(dotenv_path=env_path)

LLM_PROVIDER = os.environ.get("LLM_PROVIDER", Provider.GEMINI)

try:
    llm = build_llm(LLM_PROVIDER)
    print("=================================================================")
    print(f"Initialized Provider: {LLM_PROVIDER}")
    print(f"Model Target:         {getattr(llm, 'model', 'unknown')}")
    print("=================================================================")
except Exception as e:
    print(f"Provider setup notice: {e}")
    llm = None


Initialized Provider: gemini
Model Target:         gemini-2.0-flash


## 3. Execute LLaMEA Evolutionary Search
Run LLaMEA for a target BBOB problem. Set `RUN_EVOLUTION = True` when ready to execute.

In [ ]:

problem = BBOBProblem(problem_id=1, dim=3)

optimizer = run_evolution_for_problem(
    problem=problem,
    llm=llm,
    budget=1000,
    iterations=5,
    mode="noisy",
    noise_std=0.05,
    log=True
)
print("Evolution completed!")


## 4. Load & Analyze Experiment Results
Load `summary.json` from `logs/` directory to inspect convergence and generated code.

In [ ]:
logs_dir = src_dir / "logs"
all_experiments = [p.name for p in logs_dir.glob("*") if p.is_dir()]
print(f"Found {len(all_experiments)} logged experiment runs: {all_experiments}")

if all_experiments:
    target_exp = all_experiments[0]
    summary_file = logs_dir / target_exp / "summary.json"
    if summary_file.exists():
        with open(summary_file) as f:
            data = json.load(f)
        print(f"Analyzing Experiment: {target_exp}")
        print(f"Best Candidate Algorithm: {data.get('best_algorithm')}")
        print(f"Best Final Error: {data.get('best_final_error')}")
        
        # Convergence Plotly Chart
        history = data.get("iterations", [])
        iterations = [h["iteration"] for h in history]
        errors = [h.get("final_error", float('nan')) for h in history]
        
        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=iterations,
            y=errors,
            mode='lines+markers',
            name='Final Error',
            line=dict(color='#10B981', width=3)
        ))
        fig.update_layout(
            title=f"LLaMEA Evolutionary Convergence ({target_exp})",
            xaxis_title="Iteration",
            yaxis_title="Final Error from Optimum",
            template="plotly_white",
            height=450,
            width=850
        )
        fig.show()


## 5. Cross-Evaluation of Synthesized Algorithms
Evaluate an algorithm generated on a clean landscape against a noisy environment (or vice versa).

In [ ]:
sample_code = """
import numpy as np

class RandomSearch:
    def __call__(self, problem, budget):
        lb, ub = getattr(problem, 'lower_bound', -5.0), getattr(problem, 'upper_bound', 5.0)
        dim = getattr(problem, 'dim', 2)
        best_y = float('inf')
        for _ in range(budget):
            x = np.random.uniform(lb, ub, dim)
            y = problem(x)
            if y < best_y:
                best_y = y
        return best_y
"""

prob = BBOBProblem(problem_id=1, dim=2)

res_clean = run_cross_evaluation(code=sample_code, name="RandomSearch", problem=prob, budget=500, noise_std=0.0)
res_noisy = run_cross_evaluation(code=sample_code, name="RandomSearch", problem=prob, budget=500, noise_std=0.5)

clean_err = res_clean.get('final_error')
noisy_err = res_noisy.get('final_error')

print("=== Cross-Evaluation Results ===")
print(f"Clean Problem Error: {clean_err:.4f}" if isinstance(clean_err, float) else f"Clean Problem Error: {clean_err}")
print(f"Noisy Problem Error: {noisy_err:.4f}" if isinstance(noisy_err, float) else f"Noisy Problem Error: {noisy_err}")
